# Week 12 Supervised Learning - Classification

You will need `ny_census.csv` from Canvas. Move it into this folder or point to the absolute path of where you have it from previous weeks. 

This week we will focus on the next set of machine learning algorithms: classification. These are algorithms in which the target value, a categorical variable, is known and we use that information to "train" a model to predict on new, unseen data that does not have a target category. 

First, let's load in and prepare our data. 

In [ ]:
import pandas as pd
import geopandas as gpd 
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Let's load in our new york census data
df = pd.read_csv('ny_census.csv')
df['GEOID'] = df['GEOID'].astype(str) # We already have the GEOID but it's saved as a float

In [ ]:
df.iloc[0] # We have a lot of information about each census tract

In [ ]:
# Let's grab our outlines for plotting

year = 2020
state_fips = "36" #NY

url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state_fips}_tract.zip"
tracts = gpd.read_file(url)
tracts = tracts.to_crs(epsg=4326)
tracts.head()

# Merge with our census data
ny_df_tracts = pd.merge(left=df, right=tracts, on='GEOID', how='left')
ny_df_tracts.head()

In [ ]:
# Let's calculate density per square mile
## ALAND is in square meters right not so we need to convert it
ny_df_tracts['ALAND_sqmi'] = ny_df_tracts['ALAND']/2.59e+6
ny_df_tracts['density_sqmi'] = ny_df_tracts['TotalPopE']/ny_df_tracts['ALAND_sqmi']

In [ ]:
ny_df_tracts.density_sqmi.describe()

In [ ]:
# Let's calculate a few columns of interest

# urban/rural designation
ny_df_tracts['urban_rural'] = ny_df_tracts['density_sqmi'].apply(lambda x: 'urban' if x > 10000 else 'rural')
# percent white in each tract
ny_df_tracts['perc_white'] = ny_df_tracts['WhiteE']/ny_df_tracts['TotalPopE']
# percent below the poverty line in each tract
ny_df_tracts['perc_poverty'] =  ny_df_tracts['PovertyE']/ny_df_tracts['TotalPopE']

In [ ]:
# convert to geodataframe and plot
ny_df_tracts = gpd.GeoDataFrame(ny_df_tracts, 
                                geometry='geometry',
                                crs='EPSG:4326')
ny_df_tracts.plot(column='urban_rural', categorical=True, legend=True, cmap='coolwarm')

## 1. K-NN Classifier

The first classifier we will look at is the K-Nearest Neighbor Classifier. This works just like our spatial weights use of KNN: we are interested in the values of our _k_ closest neighbors! However, now the value we care about is the target class we are trying to predict. 

Essentially, we are looking at the _most common_ category of our _k_ nearest neighbors and assigning that class to our new point. 

In [ ]:
from sklearn import set_config
# Output dataframes instead of arrays
set_config(transform_output="pandas")

# Import the K-NN classifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Lets keep only three columns:
## Our target class (urban_rural) and two predictor columns (perc_white, MedIncomeE)
df_train = ny_df_tracts[['urban_rural', 'perc_white', 'MedIncomeE']].dropna()

In [ ]:
# What does the data look like?
sns.scatterplot(df_train, x='perc_white',y='MedIncomeE', hue='urban_rural')

In [ ]:
# As with our other machine learning algorithms, we define the setup separately here
## Let's start with 5 neighbors
knn = KNeighborsClassifier(n_neighbors=5)

In [ ]:
# Again, we "fit" the data with X data (predictors) and y data (target)
knn.fit(X=df_train[["perc_white", "MedIncomeE"]], y=df_train["urban_rural"])

In [ ]:
# Now we can predict on new data
new_obs = pd.DataFrame({"perc_white": [.9], "MedIncomeE": [60000]})
knn.predict(new_obs)

### Scaling

As we've seen before, these algorithms struggle when the scaling is very different between variables. For example, `perc_white` is only on a scale of 0 to 1 while `MedIncomeE` ranges from 0 to 250,000 in this case. This biases the results to find better separation between the classes along the `MedIncomeE` axis but not as much for `perc_white`. 

Recall the Standard Scaler and Robust Scaler from clustering...

The `scale()` or `StandardScaler` method subtracts the mean and divides by the standard deviation:

$$ z = \frac{x_i - \bar{x}}{\sigma_x}$$

This "normalizes" the variate, ensuring the re-scaled variable has a mean of zero and a variance of one. However, the variable can still be quite skewed, bimodal, etc, and insofar as the mean and vairance may be affected by outliers in a given variate, the scaling can be too dramatic. One alternative intended to handle outliers better is `robust_scale()` or `RobustScaler`, which uses the median and the interquartile range in the same fashion:

$$ z = \frac{x_i - \tilde{x}}{\lceil x \rceil_{75} - \lceil x \rceil_{25}}$$

where $\lceil x \rceil_p$ represents the value of the $p$th percentile of $x$. 

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler 
from sklearn.compose import make_column_transformer # This is a function for transforming all columns
from sklearn.compose import make_column_selector # This allows us to specify what types of columns to scale

# Create a preprocessing function (like KNN or Kmeans)
preprocessor = make_column_transformer((RobustScaler(), #run the robust scaler
                                        make_column_selector(dtype_include="number")), # only on columns that are numbers
                                       remainder="passthrough", # pass through every other column
                                       verbose_feature_names_out=False) # stops long names

In [ ]:
# fit our data 
preprocessor.fit(df_train)
# Then transform
scaled_df = preprocessor.transform(df_train)
scaled_df.head()

In [ ]:
# What does the data look like?
sns.scatterplot(scaled_df, x='perc_white',y='MedIncomeE', hue='urban_rural')

In [ ]:
## YOUR TURN 
## RUN KNN with our new scaled data and predict a new value


## 2. Train-test Split

How well does our algorithm do at classifying between our classes? We want to split our data into a training set and a testing set so that we can build a model, but save a held-out set as not bias the evaluation results. This means, the classification algorithm will learn only on the data in the training set. 


</figure>
<img src="https://python.datasciencebook.ca/_images/training_test.png" alt="drawing" width="500" style="display: block; margin: 0 auto"/>
</figure>

In [ ]:
from sklearn.model_selection import train_test_split

# We save our columns of interest and drop missing rows again
df_tmp = ny_df_tracts[['urban_rural', 'perc_white', 'MedIncomeE']].dropna()

# Get each training and testing set 
df_train, df_test = train_test_split(df_tmp, train_size=0.75)


In [ ]:
# Optional: keep the class balance the same for training and testing
df_train, df_test = train_test_split(df_tmp, train_size=0.75, stratify=df_tmp["urban_rural"])
print(df_tmp['urban_rural'].value_counts(normalize=True))
print(df_train['urban_rural'].value_counts(normalize=True))
print(df_test['urban_rural'].value_counts(normalize=True))


### Pipelines

When we start to have training and testing sets, or want to build algorithms that will then be used on new data later, it becomes helpful to institute **pipelines**. These are akin to multi-step functions that transform your data in the same way every single time. 

For example, we want to preprocess all data that comes into our model the same way every single time. 

In [ ]:
# Recall our preprocessor
preprocessor = make_column_transformer((RobustScaler(), #run the robust scaler
                                        make_column_selector(dtype_include="number")), # only on columns that are numbers
                                       remainder="passthrough", # pass through every other column
                                       verbose_feature_names_out=False) # stops long names

In [ ]:
# And recall our KNN model
knn = KNeighborsClassifier(n_neighbors=5)

In [ ]:
from sklearn.pipeline import make_pipeline

# Now we can build them into one pipeline so data always gets preprocessed the same way 
knn_pipeline = make_pipeline(preprocessor, knn)

knn_pipeline

In [ ]:
# Let's fit our model
knn_pipeline.fit(X=df_train[["perc_white", "MedIncomeE"]], y=df_train["urban_rural"])

knn_pipeline

In [ ]:
# Now we can predict on our test data using the same pipeline
predicted = knn_pipeline.predict(df_test[["perc_white", "MedIncomeE"]])
df_test['predicted'] = predicted # Add new column
df_test[["urban_rural", "predicted"]].head(10) # Show side by side

In [ ]:
# How well do we do? This returns "accuracy"
knn_pipeline.score(df_test[['perc_white', 'MedIncomeE']], df_test['urban_rural'])

## 3. Evaluation Metrics

There are a few standard evaluation metrics we can compute with classification tasks. They all stem from four main values:
* True positive: classified as category 1, actually is category 1
* False positive: classified as category 1, actually is category 0
* True negative: classified as category 0, actually is category 0
* False negative: classified as category 0, actually is category 1

### Confusion Matrix

We can see these four values in a **confusion matrix**.


In [ ]:
# Method 1: through pandas crosstab
pd.crosstab(df_test["urban_rural"], df_test["predicted"])

In [ ]:
# Method 2: through sklearn.metrics
from sklearn.metrics import confusion_matrix
confusion_matrix(df_test["urban_rural"], df_test["predicted"])

In [ ]:
# (This one is nice because you can get to the four values slightly easier)
tn, fp, fn, tp = confusion_matrix(df_test["urban_rural"], df_test["predicted"]).ravel().tolist()
print(tn)
print(fp)

### Precision, Recall, Accuracy, F1

From these four values, we can get to numbers that we care about. 
* Precision shows how many labeled items are correct
  
$$\frac{TP}{TP + FP}$$
* Recall shows how many actual items are labeled
$$\frac{TP}{TP + FN}$$
</figure>
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/Precisionrecall.svg/500px-Precisionrecall.svg.png" alt="drawing" width="200" style="display: block; margin: 0 auto"/>
</figure>


* Accuracy is the total correct over the total possible (prone to bias)
$$\frac{\sum TP+FN}{\sum TP+FN+FP+TN }$$

* F1 Score is the harmonic mean between precision and recall (more reflective of model performance)
$$\frac{2*precision*recall}{precision+recall}$$

In [ ]:
# Method 1: getting each from sklean individually
from sklearn.metrics import recall_score, precision_score, accuracy_score, f1_score

print(precision_score(df_test["urban_rural"], df_test["predicted"], pos_label='urban'))
print(recall_score(df_test["urban_rural"], df_test["predicted"], pos_label='urban'))

In [ ]:
print(accuracy_score(df_test["urban_rural"], df_test["predicted"]))
print(f1_score(df_test["urban_rural"], df_test["predicted"], pos_label='urban'))

In [ ]:
# Method 2 classification report
from sklearn.metrics import classification_report
print(classification_report(df_test["urban_rural"], df_test["predicted"]))

In [ ]:
## YOUR TURN 
## Change the pipeline to have a knn model with a different k (higher? lower?)



## Use our train-test split to fit and predict


## Compute the precision, recall, and F1 scores of this model 
